In [ ]:
from collections import defaultdict
import numpy as np


# ============================================================
# Input files
# ============================================================

GUIDE_FILE = (
    "filtered_full_guide_seqs.txt"
)

TARGET_FILE = (
    "filtered_target_seqs_rc.txt"
)

ACTIVITY_FILE = (
    "filtered_log_transformed_activity.txt"
)


# ============================================================
# Output files
# ============================================================

OUTPUT_GUIDE_FILE = (
    "decreased_relative_activity_filtered_full_guide_seqs.txt"
)

OUTPUT_TARGET_FILE = (
    "decreased_relative_activity_filtered_target_seqs_rc.txt"
)

OUTPUT_ACTIVITY_FILE = (
    "decreased_relative_activity_filtered_log_transformed_activity.txt"
)


# ============================================================
# Helper functions
# ============================================================

def read_nonempty_lines(file_path):
    """
    Read all nonempty lines from a text file.
    """
    with open(file_path, "r") as file:
        return [
            line.strip()
            for line in file
            if line.strip()
        ]


def normalize_rna_sequence(sequence):
    """
    Normalize an RNA guide sequence.

    T is converted to U in case the guide file uses DNA notation.
    """
    return sequence.strip().upper().replace("T", "U")


def normalize_dna_sequence(sequence):
    """
    Normalize a DNA target sequence.

    U is converted to T in case the target file contains RNA notation.
    """
    return sequence.strip().upper().replace("U", "T")


def reverse_complement_rna_to_dna(rna_sequence):
    """
    Generate the reverse-complementary DNA sequence for an RNA guide.
    """
    rna_sequence = normalize_rna_sequence(rna_sequence)

    complement = {
        "A": "T",
        "U": "A",
        "C": "G",
        "G": "C"
    }

    invalid_bases = set(rna_sequence) - set(complement)

    if invalid_bases:
        raise ValueError(
            f"Invalid nucleotide(s) {sorted(invalid_bases)} "
            f"found in RNA guide: {rna_sequence}"
        )

    return "".join(
        complement[base]
        for base in reversed(rna_sequence)
    )


# ============================================================
# 1. Read the three input files
# ============================================================

guide_lines = read_nonempty_lines(GUIDE_FILE)
target_lines = read_nonempty_lines(TARGET_FILE)
activity_lines = read_nonempty_lines(ACTIVITY_FILE)


if not (
    len(guide_lines)
    == len(target_lines)
    == len(activity_lines)
):
    raise ValueError(
        "The three input files do not contain the same number of "
        "nonempty lines.\n"
        f"Guide file:    {len(guide_lines)} lines\n"
        f"Target file:   {len(target_lines)} lines\n"
        f"Activity file: {len(activity_lines)} lines"
    )


# Convert activity values to floats
activities = []

for line_number, value in enumerate(activity_lines, start=1):
    try:
        activity = float(value)
    except ValueError as error:
        raise ValueError(
            f"The activity value on line {line_number} is not numeric: "
            f"{value}"
        ) from error

    if not np.isfinite(activity):
        raise ValueError(
            f"The activity value on line {line_number} is not finite: "
            f"{value}"
        )

    activities.append(activity)


# ============================================================
# 2. Globally normalize the entire activity list to [0, 1]
# ============================================================

minimum_activity = min(activities)
maximum_activity = max(activities)

if maximum_activity == minimum_activity:
    raise ValueError(
        "All activity values are identical, so min-max normalization "
        "cannot be performed."
    )


normalized_activities = [
    (activity - minimum_activity)
    / (maximum_activity - minimum_activity)
    for activity in activities
]


# ============================================================
# 3. Detect perfectly matched RNA-guide/DNA-target pairs
# ============================================================

guide_20nt_list = []
target_20nt_list = []
expected_target_20nt_list = []
is_exact_match_list = []


for line_number, (guide, target) in enumerate(
    zip(guide_lines, target_lines),
    start=1
):
    guide_rna = normalize_rna_sequence(guide)
    target_dna = normalize_dna_sequence(target)

    if len(guide_rna) < 20:
        raise ValueError(
            f"Guide on line {line_number} is shorter than 20 nt: "
            f"{guide_rna}"
        )

    if len(target_dna) < 20:
        raise ValueError(
            f"Target on line {line_number} is shorter than 20 nt: "
            f"{target_dna}"
        )

    # Extract the first 20 nt of the RNA guide
    guide_20nt = guide_rna[0:20]

    # Generate the expected reverse-complementary DNA target
    expected_target_20nt = reverse_complement_rna_to_dna(
        guide_20nt
    )

    # Extract the final 20 nt of the DNA target
    target_20nt = target_dna[-20:]

    is_exact_match = (
        target_20nt == expected_target_20nt
    )

    guide_20nt_list.append(guide_20nt)
    target_20nt_list.append(target_20nt)
    expected_target_20nt_list.append(expected_target_20nt)
    is_exact_match_list.append(is_exact_match)


# ============================================================
# 4. Find the normalized perfect-match activity for each guide
# ============================================================

matched_activities_by_guide = defaultdict(list)


for guide_20nt, normalized_activity, is_exact_match in zip(
    guide_20nt_list,
    normalized_activities,
    is_exact_match_list
):
    if is_exact_match:
        matched_activities_by_guide[guide_20nt].append(
            normalized_activity
        )


# If multiple perfect-match rows exist for one guide,
# use their mean normalized activity.
reference_activity_by_guide = {
    guide_20nt: float(np.mean(activity_values))
    for guide_20nt, activity_values
    in matched_activities_by_guide.items()
}


# Guides lacking a perfect-match row cannot be normalized.
guides_without_perfect_match = {
    guide_20nt
    for guide_20nt in set(guide_20nt_list)
    if guide_20nt not in reference_activity_by_guide
}


# A reference normalized activity of zero would cause division by zero.
guides_with_zero_reference = {
    guide_20nt
    for guide_20nt, reference_activity
    in reference_activity_by_guide.items()
    if np.isclose(reference_activity, 0.0)
}


# Remove all rows belonging to either type of invalid guide.
removed_guides = (
    guides_without_perfect_match
    | guides_with_zero_reference
)

valid_guides = (
    set(guide_20nt_list) - removed_guides
)


# ============================================================
# 5. Calculate decreased relative activity
#
# Perfect pair:
#     0
#
# Mismatched pair:
#     (A_perfect - A_mismatch) / A_perfect
#
# Activities here are globally min-max normalized to [0, 1].
# ============================================================

filtered_guide_lines = []
filtered_target_lines = []
decreased_relative_activities = []

removed_line_numbers = []


for line_number, (
    original_guide,
    original_target,
    guide_20nt,
    normalized_activity,
    is_exact_match
) in enumerate(
    zip(
        guide_lines,
        target_lines,
        guide_20nt_list,
        normalized_activities,
        is_exact_match_list
    ),
    start=1
):
    # Remove all samples belonging to a guide that has no usable
    # perfectly matched reference.
    if guide_20nt not in valid_guides:
        removed_line_numbers.append(line_number)
        continue

    matched_activity = reference_activity_by_guide[guide_20nt]

    if is_exact_match:
        decreased_relative_activity = 0.0
    else:
        decreased_relative_activity = (
            matched_activity - normalized_activity
        ) / matched_activity

    filtered_guide_lines.append(original_guide)
    filtered_target_lines.append(original_target)
    decreased_relative_activities.append(
        decreased_relative_activity
    )


# ============================================================
# 6. Write the three synchronized output files
# ============================================================

with open(OUTPUT_GUIDE_FILE, "w") as output_file:
    for guide in filtered_guide_lines:
        output_file.write(guide + "\n")


with open(OUTPUT_TARGET_FILE, "w") as output_file:
    for target in filtered_target_lines:
        output_file.write(target + "\n")


with open(OUTPUT_ACTIVITY_FILE, "w") as output_file:
    for value in decreased_relative_activities:
        output_file.write(f"{value:.10g}\n")


# ============================================================
# 7. Print a processing summary
# ============================================================

number_original_samples = len(guide_lines)
number_removed_samples = len(removed_line_numbers)
number_remaining_samples = len(filtered_guide_lines)

number_original_guides = len(set(guide_20nt_list))
number_removed_guides = len(removed_guides)
number_remaining_guides = len(valid_guides)

number_exact_matches_retained = sum(
    is_exact_match
    for guide_20nt, is_exact_match in zip(
        guide_20nt_list,
        is_exact_match_list
    )
    if guide_20nt in valid_guides
)

number_mismatches_retained = (
    number_remaining_samples
    - number_exact_matches_retained
)


print("Processing completed.")
print()

print("Original activity range:")
print(f"  Minimum: {minimum_activity}")
print(f"  Maximum: {maximum_activity}")
print()

print(f"Original samples: {number_original_samples}")
print(f"Removed samples:  {number_removed_samples}")
print(f"Remaining samples: {number_remaining_samples}")
print()

print(f"Original distinct guides: {number_original_guides}")
print(f"Guides removed:           {number_removed_guides}")
print(f"Guides retained:          {number_remaining_guides}")
print()

print(
    "Perfect-match samples retained: "
    f"{number_exact_matches_retained}"
)
print(
    "Mismatch samples retained:      "
    f"{number_mismatches_retained}"
)
print()

print("Output files:")
print(f"  {OUTPUT_GUIDE_FILE}")
print(f"  {OUTPUT_TARGET_FILE}")
print(f"  {OUTPUT_ACTIVITY_FILE}")


# ============================================================
# 8. Display removed guides
# ============================================================

if guides_without_perfect_match:
    print()
    print(
        "Guides removed because no perfectly matched reference "
        "was found:"
    )

    for guide_20nt in sorted(guides_without_perfect_match):
        affected_lines = [
            line_number
            for line_number, current_guide in enumerate(
                guide_20nt_list,
                start=1
            )
            if current_guide == guide_20nt
        ]

        print(
            f"  Guide: {guide_20nt} | "
            f"Removed lines: {affected_lines}"
        )


if guides_with_zero_reference:
    print()
    print(
        "Guides removed because their normalized perfect-match "
        "activity was zero:"
    )

    for guide_20nt in sorted(guides_with_zero_reference):
        affected_lines = [
            line_number
            for line_number, current_guide in enumerate(
                guide_20nt_list,
                start=1
            )
            if current_guide == guide_20nt
        ]
